In [83]:
import zipfile 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import *
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


## Data Import

In [ ]:
d = pd.read_csv("EVChargingStationUsage.csv", low_memory=False)
sites = ["hamilton","cambridge","webster","sherman","bryant","mpl","rinconada","thompson","high"]

d["site"] = d["Station Name"].str.lower().str.extract("(" + "|".join(sites) + ")", expand=False)
print(d["site"].value_counts(dropna=False))

site
bryant       47100
high         39901
hamilton     38609
webster      37585
cambridge    32879
mpl          28326
rinconada    18264
thompson     16712
sherman         39
Name: count, dtype: int64


In [85]:
df = d[["site","Energy (kWh)","Charging Time (hh:mm:ss)","Start Date"]].copy()
df = df[df["site"] != "sherman"]          # Sherman opened Dec 2020 -> no training history
df.columns = ["site","energy","charging_time","start_time"]

In [86]:
date_columns = [
    "start_time"
]
duration_columns = [
    "charging_time",
]

for column in date_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce", format="mixed")
for column in duration_columns:
    df[column] = pd.to_timedelta(df[column], errors="coerce")

columns_to_check = date_columns + duration_columns
faulty_rows = df[columns_to_check].isna().any(axis=1)
print(f"Dropping {faulty_rows.sum()} faulty observations")
df = df.loc[~faulty_rows].reset_index(drop=True)

Dropping 0 faulty observations


### From sessions to hourly energy per site
The task is hourly consumption per station, so each session's energy is spread over the hours it was charging. Hours with no charging are 0, and hours before a site opened or after it stopped reporting are missing.

In [88]:
# Each session's energy is spread over the clock hours it was actually charging
# (start -> start + charging time), proportional to the minutes charged in each hour.
start = df["start_time"]
end = start + df["charging_time"]
first_hour = start.dt.floor("h")
n_hours = ((end.dt.floor("h") - first_hour) // pd.Timedelta("1h")).astype(int) + 1

rep = np.repeat(np.arange(len(df)), n_hours)                    # one row per (session, hour)
k = np.concatenate([np.arange(n) for n in n_hours])
hour = first_hour.values[rep] + k * np.timedelta64(1, "h")
overlap = (np.minimum(end.values[rep], hour + np.timedelta64(1, "h"))
           - np.maximum(start.values[rep], hour)) / np.timedelta64(1, "s")
overlap = np.maximum(overlap, 0)
total = np.bincount(rep, weights=overlap, minlength=len(df))
share = np.where(total[rep] > 0, overlap / np.where(total[rep] > 0, total[rep], 1), k == 0)  # 0-length sessions -> start hour

long = pd.DataFrame({"site": df["site"].values[rep], "hour": hour,
                     "kWh": df["energy"].values[rep] * share})
hourly = long.pivot_table(index="hour", columns="site", values="kWh", aggfunc="sum")
hourly = hourly.reindex(pd.date_range(hourly.index.min().floor("D"),
                                      hourly.index.max().floor("D") + pd.Timedelta("23h"), freq="h"))
hourly = hourly.fillna(0)

# Before a site opened (or after it closed / stopped reporting) its hours are missing, not zero
opened = df.groupby("site")["start_time"].min().dt.floor("D")
closed = df.groupby("site")["start_time"].max().dt.floor("D") + pd.Timedelta("1D")
for site in hourly.columns:
    hourly.loc[(hourly.index < opened[site]) | (hourly.index >= closed[site]), site] = np.nan
print(pd.DataFrame({"opened": opened.dt.date, "last data": closed.dt.date}))

print(hourly.shape, "| energy check:", round(hourly.sum().sum() - df["energy"].sum(), 6))
hourly.tail()

               opened   last data
site                             
bryant     2011-10-10  2021-01-01
cambridge  2016-01-14  2021-01-01
hamilton   2011-07-29  2020-11-05
high       2011-09-02  2021-01-01
mpl        2014-09-24  2021-01-01
rinconada  2015-01-02  2020-07-06
thompson   2016-01-14  2021-01-01
webster    2015-07-29  2021-01-01
(82656, 8) | energy check: 0.0


site,bryant,cambridge,hamilton,high,mpl,rinconada,thompson,webster
2020-12-31 19:00:00,0.0,0.027605,NaN,0.0,10.209486,NaN,0.0,3.271163
2020-12-31 20:00:00,0.0,0.000000,NaN,0.0,5.775146,NaN,0.0,3.074711
2020-12-31 21:00:00,0.0,0.000000,NaN,0.0,5.775146,NaN,0.0,2.188169
2020-12-31 22:00:00,0.0,0.000000,NaN,0.0,3.484338,NaN,0.0,0.000000
2020-12-31 23:00:00,0.0,0.000000,NaN,0.0,0.000000,NaN,0.0,0.000000


### Train/test split
The first 70% of calendar days are used for training and the rest for testing. The split is chronological with no shuffling, as the project description requires.

In [89]:
# Chronological 70/30 split on calendar days (same cut-off for all sites, no shuffling)
days = hourly.index.normalize().unique()
cutoff = days[int(0.7 * len(days))]
train = hourly[hourly.index < cutoff]
test = hourly[hourly.index >= cutoff]
print(f"Train: {train.index.min().date()} -> {train.index.max().date()} ({len(train)//24} days)")
print(f"Test:  {test.index.min().date()} -> {test.index.max().date()} ({len(test)//24} days)")

Train: 2011-07-29 -> 2018-03-03 (2410 days)
Test:  2018-03-04 -> 2020-12-31 (1034 days)


### Baseline Model

In [101]:
# Baseline: the "average Monday" (etc.) per site, estimated on the training set only.
# profile[(weekday, hour), site] = mean kWh in that hour at that site
profile = train.groupby([train.index.dayofweek, train.index.hour]).mean()
profile.index.names = ["weekday", "hour"]

def baseline(train, test):
    """Predict every test hour with the training-set average for that site, weekday and hour."""
    profile = train.groupby([train.index.dayofweek, train.index.hour]).mean()
    keys = pd.MultiIndex.from_arrays([test.index.dayofweek, test.index.hour])
    pred = profile.reindex(keys)
    pred.index = test.index
    return pred[test.columns]

y_pred_base = baseline(train, test)

In [91]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def scores(y_true, y_pred, name):
    """MAE / RMSE / R^2 over all (site, hour) pairs where the site existed."""
    t, p = y_true.to_numpy().ravel(), y_pred[y_true.columns].to_numpy().ravel()
    ok = ~np.isnan(t) & ~np.isnan(p)                 # only hours where the site was open
    return pd.Series({"MAE": mean_absolute_error(t[ok], p[ok]),
                      "RMSE": np.sqrt(mean_squared_error(t[ok], p[ok])),
                      "R2": r2_score(t[ok], p[ok]),
                      "n": ok.sum()}, name=name)

pre_covid = test.index < "2020-03-01"
base_scores = pd.DataFrame([
    scores(test, y_pred_base, "Baseline, full test"),
    scores(test[pre_covid], y_pred_base[pre_covid], "Baseline, test before Mar 2020"),
])
display(base_scores.round(3))

per_site = pd.DataFrame({s: scores(test[[s]], y_pred_base[[s]], s) for s in test.columns}).T
per_site["mean kWh/h (test)"] = test.mean()
per_site.round(3)

,MAE,RMSE,R2,n
"Baseline, full test",4.093,6.091,0.214,192864.0
"Baseline, test before Mar 2020",4.208,6.415,0.256,139776.0


,MAE,RMSE,R2,n,mean kWh/h (test)
bryant,5.088,7.945,0.227,24816.0,6.364
cambridge,4.247,6.536,0.300,24816.0,5.965
hamilton,2.733,3.493,0.365,23448.0,3.988
high,4.534,7.354,0.193,24816.0,5.976
mpl,3.650,4.541,-0.269,24816.0,3.138
rinconada,3.292,4.084,-0.283,20520.0,2.515
thompson,3.797,6.189,0.151,24816.0,4.461
webster,5.189,6.646,0.158,24816.0,6.024


#### Baseline test scores

Scores are in kWh per site-hour, over all hours where the site was open (8 sites, test period 4 Mar 2018 – 31 Dec 2020).

| Test set | MAE | RMSE | $R^2$ |
|---|---|---|---|
| Full test period | 4.09 | 6.09 | 0.21 |
| Test before Mar 2020 (pre-COVID) | 4.21 | 6.42 | 0.26 |

Why the average profile does poorly:
- **Growth:** usage at Bryant, High and Thompson grew a lot after the training years, so the baseline under-predicts the peaks.
- **Shifted patterns:** MPL and Rinconada (the library sites) have negative $R^2$. The training average still expects evening and night use that no longer happens in the test period.
- **COVID:** 2020 usage collapses. MAE drops only because the true values are small.

Models that use recent lags (yesterday, same day last week) should handle all three, which is the comparison to make next.

## Prediction Models

**Approach:** the forecast is framed as a regression problem. There is one row per site and hour of the next day, with target $y_{s,t}$ = kWh at site $s$ in hour $t$. The rows from all sites are stacked into one table and a single model is trained on it.

The time-series structure enters through **lagged features**: the same hour yesterday, the same hour last week, and recent averages. These are combined with calendar and capacity features. Every feature is computed only from data up to midnight before the predicted day, so the model can be used exactly as the task describes. Each test day is predicted from its own previous days, so this is a rolling one-day-ahead forecast.

**Models compared**, all on the same test rows as the baseline:
- two naive benchmarks: "same hour yesterday" and "same hour last week"
- **Ridge regression** (linear)
- **Random forest** (non-linear)
- **Gradient boosting** (non-linear)

Hyperparameters are chosen on a validation period: the last 15% of the training days. The test set is only used for the final scores.

### Feature engineering
| Group | Features |
|---|---|
| Lags | `lag_24`, `lag_48`, `lag_168` (same hour 1, 2 and 7 days earlier), `mean7_same_hour` (mean of that hour over the last 7 days) |
| Recent level | `yesterday_total`, `mean7_daily_total` |
| Capacity | `active_chargers`: chargers at the site with at least one session in the previous 7 days |
| Calendar | `hour`, `weekday`, `weekend`, `holiday` (US federal), `month` |
| Site | `site` |

In [93]:
# All features are known at midnight before the day being predicted (day D).
# For a target hour t on day D, "lag 24" is the same hour on day D-1, "lag 168" the same hour on D-7, etc.
from pandas.tseries.holiday import USFederalHolidayCalendar

lag = {f"lag_{h}": hourly.shift(h) for h in [24, 48, 168]}
lag["mean7_same_hour"] = sum(hourly.shift(24 * k) for k in range(1, 8)) / 7

daily = hourly.resample("D").sum(min_count=24)                    # kWh per site per day
to_hours = lambda x: x.reindex(hourly.index, method="ffill")      # broadcast a daily value to its 24 hours
lag["yesterday_total"] = to_hours(daily.shift(1))
lag["mean7_daily_total"] = to_hours(daily.rolling(7).mean().shift(1))

# Capacity: number of chargers at the site that had at least one session during the previous 7 days
chg = d[d["site"].isin(hourly.columns)].assign(day=pd.to_datetime(d["Start Date"], format="mixed").dt.floor("D"))
active = (chg.groupby(["day", "site", "Station Name"]).size().gt(0).unstack(["site", "Station Name"])
             .reindex(daily.index).fillna(False).astype(int))
active = active.rolling(7, min_periods=1).max().T.groupby(level="site").sum().T   # chargers active in last 7 days
lag["active_chargers"] = to_hours(active.shift(1).reindex(columns=hourly.columns))

# Long format: one row per (hour, site). Keep NaNs while stacking (argument differs between pandas 2 and 3).
def stack(frame):
    return frame.stack(dropna=False) if pd.__version__.startswith(("1.", "2.")) else frame.stack()

long = stack(hourly).rename("y").to_frame()
for name, frame in lag.items():
    long[name] = stack(frame)
long.index.names = ["time", "site"]
long = long.reset_index()

t = long["time"]
holidays = USFederalHolidayCalendar().holidays(t.min(), t.max())
long["hour"] = t.dt.hour
long["weekday"] = t.dt.dayofweek
long["weekend"] = (long["weekday"] >= 5).astype(int)
long["holiday"] = t.dt.normalize().isin(holidays).astype(int)
long["month"] = t.dt.month

n_before = len(long)
long = long.dropna().reset_index(drop=True)      # site closed, or not enough history yet for the lags
print(f"{len(long):,} rows ({n_before - len(long):,} dropped: site not open or lags not yet available)")
long.head()

480,600 rows (180,648 dropped: site not open or lags not yet available)


,time,site,y,lag_24,lag_48,lag_168,mean7_same_hour,yesterday_total,mean7_daily_total,active_chargers,hour,weekday,weekend,holiday,month
0,2011-08-05 00:00:00,hamilton,0.0,2.800741,0.0,0.0,0.415333,29.741112,21.47193,2.0,0,4,0,0,8
1,2011-08-05 01:00:00,hamilton,0.0,1.970633,0.0,0.0,0.281519,29.741112,21.47193,2.0,1,4,0,0,8
2,2011-08-05 02:00:00,hamilton,0.0,0.000000,0.0,0.0,0.000000,29.741112,21.47193,2.0,2,4,0,0,8
3,2011-08-05 03:00:00,hamilton,0.0,0.000000,0.0,0.0,0.000000,29.741112,21.47193,2.0,3,4,0,0,8
4,2011-08-05 04:00:00,hamilton,0.0,0.000000,0.0,0.0,0.223169,29.741112,21.47193,2.0,4,4,0,0,8


### Design matrices and train/test rows
Tree models take the categorical variables as integer codes. Ridge regression needs them one-hot encoded, including hour × weekend and hour × site interactions so it can represent each site's daily shape. The train/test cut-off is the same as for the baseline.

In [94]:
lag_cols = ["lag_24", "lag_48", "lag_168", "mean7_same_hour", "yesterday_total", "mean7_daily_total", "active_chargers"]
cal_cols = ["hour", "weekday", "weekend", "holiday", "month"]

is_train = long["time"] < cutoff
is_test = ~is_train
y_train, y_test = long.loc[is_train, "y"].values, long.loc[is_test, "y"].values

# Trees: categories as integer codes are fine (trees split on them directly)
X_tree = long[lag_cols + cal_cols].copy()
X_tree["site"] = long["site"].astype("category").cat.codes

# Linear model: one-hot categoricals + hour x weekend and hour x site interactions so it can represent the daily shapes
X_lin = pd.concat([
    long[lag_cols + ["holiday"]],
    pd.get_dummies(long["weekday"], prefix="wd", drop_first=True),
    pd.get_dummies(long["month"], prefix="m", drop_first=True),
    pd.get_dummies(long["hour"].astype(str) + "_" + long["weekend"].astype(str), prefix="h_we", drop_first=True),
    pd.get_dummies(long["site"] + "_" + long["hour"].astype(str), prefix="site_h", drop_first=True),
], axis=1).astype(float)

print("Tree features:", X_tree.shape[1], "| Linear features:", X_lin.shape[1])
print(f"Train rows: {is_train.sum():,} | Test rows: {is_test.sum():,}")

Tree features: 13 | Linear features: 263
Train rows: 287,736 | Test rows: 192,864


### Model selection on a validation period

In [95]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

# Validation = last 15% of the TRAINING days. The test set is not touched while tuning.
train_days = long.loc[is_train, "time"].dt.normalize().unique()
val_start = pd.Timestamp(np.sort(train_days)[int(0.85 * len(train_days))])
fit_idx = is_train & (long["time"] < val_start)
val_idx = is_train & (long["time"] >= val_start)

# (model name, setting label, model) - a small grid per model to keep run time reasonable
candidates = [
    *[("Ridge regression", f"alpha={a}", make_pipeline(StandardScaler(), Ridge(alpha=a))) for a in [1, 100, 10_000]],
    *[("Random forest", f"min_samples_leaf={l}",
       RandomForestRegressor(n_estimators=100, min_samples_leaf=l, max_features=0.5,
                             max_samples=0.3, n_jobs=-1, random_state=0)) for l in [5, 20]],
    *[("Gradient boosting", f"learning_rate={lr}, max_leaf_nodes={n}",
       HistGradientBoostingRegressor(learning_rate=lr, max_leaf_nodes=n, max_iter=500,
                                     early_stopping=False, random_state=0)) for lr, n in [(0.05, 31), (0.1, 63)]],
]
best, tuning = {}, []
for name, setting, m in candidates:
    X = X_lin if name == "Ridge regression" else X_tree
    m.fit(X[fit_idx], long.loc[fit_idx, "y"])
    mae = mean_absolute_error(long.loc[val_idx, "y"], m.predict(X[val_idx]).clip(0))
    tuning.append({"Model": name, "Setting": setting, "Validation MAE": mae})
    if name not in best or mae < best[name][1]:
        best[name] = (m, mae)
print(f"Fit: {long.loc[fit_idx, 'time'].min().date()} -> {val_start.date()} | Validation: {val_start.date()} -> {cutoff.date()}")
pd.DataFrame(tuning).round(3)

Fit: 2011-08-05 -> 2017-03-08 | Validation: 2017-03-08 -> 2018-03-04


,Model,Setting,Validation MAE
0,Ridge regression,alpha=1,2.977
1,Ridge regression,alpha=100,2.977
2,Ridge regression,alpha=10000,2.982
3,Random forest,min_samples_leaf=5,2.839
4,Random forest,min_samples_leaf=20,2.823
5,Gradient boosting,"learning_rate=0.05, max_leaf_nodes=31",2.820
6,Gradient boosting,"learning_rate=0.1, max_leaf_nodes=63",2.855


### Final fit on the full training period and test predictions

In [96]:
from sklearn.base import clone

# Refit the chosen settings on the full training period and predict the test period
predictions = {}
for name, (m, _) in best.items():
    X = X_lin if name == "Ridge regression" else X_tree
    final = clone(m).fit(X[is_train], y_train)
    predictions[name] = final.predict(X[is_test]).clip(0)       # energy can't be negative
    best[name] = (final, best[name][1])

# Benchmarks on exactly the same test rows
test_rows = long[is_test]
base_long = y_pred_base.stack().rename("base").reset_index()
base_long.columns = ["time", "site", "base"]
predictions = {
    "Baseline (average weekday profile)": test_rows.merge(base_long, on=["time", "site"], how="left")["base"].values,
    "Naive: same hour yesterday": test_rows["lag_24"].values,
    "Naive: same hour last week": test_rows["lag_168"].values,
    **predictions,
}

### Results

In [97]:
from sklearn.metrics import mean_squared_error, r2_score

def score_row(y, p):
    return {"MAE": mean_absolute_error(y, p), "RMSE": np.sqrt(mean_squared_error(y, p)), "R2": r2_score(y, p)}

pre = (test_rows["time"] < "2020-03-01").values
rows = []
for name, p in predictions.items():
    full, pc = score_row(y_test, p), score_row(y_test[pre], p[pre])
    rows.append({"Model": name,
                 "MAE": full["MAE"], "RMSE": full["RMSE"], "R2": full["R2"],
                 "MAE pre-COVID": pc["MAE"], "RMSE pre-COVID": pc["RMSE"], "R2 pre-COVID": pc["R2"]})
summary = pd.DataFrame(rows).set_index("Model")
summary["MAE vs baseline"] = (summary["MAE"] / summary.loc["Baseline (average weekday profile)", "MAE"] - 1).map("{:+.0%}".format)
summary["RMSE vs baseline"] = (summary["RMSE"] / summary.loc["Baseline (average weekday profile)", "RMSE"] - 1).map("{:+.0%}".format)
print("Test period 4 Mar 2018 - 31 Dec 2020, kWh per site-hour. Pre-COVID = test hours before 1 Mar 2020.")
summary.round(3)

Test period 4 Mar 2018 - 31 Dec 2020, kWh per site-hour. Pre-COVID = test hours before 1 Mar 2020.


,MAE,RMSE,R2,MAE pre-COVID,RMSE pre-COVID,R2 pre-COVID,MAE vs baseline,RMSE vs baseline
Model,,,,,,,,
Baseline (average weekday profile),4.093,6.091,0.214,4.208,6.415,0.256,+0%,+0%
Naive: same hour yesterday,3.758,6.088,0.215,4.273,6.644,0.202,-8%,-0%
Naive: same hour last week,3.463,5.625,0.330,3.887,6.069,0.335,-15%,-8%
Ridge regression,3.154,4.540,0.564,3.518,4.934,0.560,-23%,-25%
Random forest,2.992,4.370,0.596,3.327,4.744,0.593,-27%,-28%
Gradient boosting,2.974,4.335,0.602,3.289,4.684,0.604,-27%,-29%


#### Summary of results

Test period 4 Mar 2018 – 31 Dec 2020, in kWh per site-hour. Percentages are the change relative to the baseline.

| Model | MAE | RMSE | $R^2$ | MAE vs baseline |
|---|---|---|---|---|
| Baseline (average weekday profile) | 4.09 | 6.09 | 0.21 | – |
| Naive: same hour yesterday | 3.76 | 6.09 | 0.22 | −8% |
| Naive: same hour last week | 3.46 | 5.63 | 0.33 | −15% |
| Ridge regression | 3.15 | 4.54 | 0.56 | −23% |
| Random forest | 2.99 | 4.37 | 0.60 | −27% |
| **Gradient boosting** | **2.98** | **4.33** | **0.60** | **−27%** |

- All three models beat the baseline at every site, and the ranking is the same before and after COVID.
- Gradient boosting is best overall, with the random forest very close. The non-linear models gain about 5% in MAE over Ridge.
- The most important feature by far is `mean7_same_hour`: the recent level at that hour. This is what the historical-average baseline is missing, since it can't follow growth, changed patterns at the library sites (MPL, Rinconada) or the 2020 drop.
- The largest errors are at midday (10:00–14:00), when demand is highest and most variable.